# Rilevamento Aree Umide tramite C-NN su Dati multispettrali Sentinel-2

Questo notebook dimostra come:
1. Accedere ai dati Sentinel-2 da Planetary Computer
2. Preprocessare le immagini satellitari
3. Applicare il modello (U-Net) per rilevare aree umide
4. Visualizzare e analizzare i risultati

L'intera pipeline è stata progettata per rilevare efficacemente aree umide come laghi, fiumi, e altre superfici d'acqua utilizzando immagini multispettrali di Sentinel-2.

## 1. Installazione delle dipendenze

In [ ]:
# Installiamo le dipendenze necessarie
!pip install planetary-computer pystac-client rioxarray matplotlib torch torchvision numpy scipy scikit-image geopandas earthpy folium rasterio gdown segmentation-models-pytorch

## 2. Importazione delle librerie

In [ ]:
import pystac_client
import planetary_computer
import rioxarray
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import geopandas as gpd
import folium
from shapely.geometry import box
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from skimage.transform import resize
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.transform import from_bounds
import gdown

# Configurazione di base
plt.rcParams['figure.figsize'] = (14, 8)

## 3. Connessione a Planetary Computer

In [ ]:
# Connessione al catalogo STAC di Planetary Computer
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

print("Connessione al catalogo STAC di Planetary Computer stabilita con successo.")

## 4. Definizione dell'area di interesse (AOI)

In [ ]:
# Definiamo un'area di interesse (AOI)
# Parametri predefiniti: Lago Sirino, Italia
lon_min, lat_min, lon_max, lat_max = 15.7, 40.0, 15.9, 40.2

# Creiamo un GeoDataFrame per l'AOI
aoi_geom = box(lon_min, lat_min, lon_max, lat_max)
aoi = gpd.GeoDataFrame(geometry=[aoi_geom], crs="EPSG:4326")

# Visualizzazione dell'AOI
m = folium.Map(location=[(lat_min + lat_max)/2, (lon_min + lon_max)/2], zoom_start=10)
folium.GeoJson(aoi).add_to(m)
m

## 5. Ricerca di immagini Sentinel-2

In [ ]:
# Ricerchiamo le immagini Sentinel-2 L2A
# Definiamo un intervallo di tempo recente con poca copertura nuvolosa
now = datetime.now()
start_date = now - timedelta(days=90)
end_date = now

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[lon_min, lat_min, lon_max, lat_max],
    datetime=f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}",
    query={"eo:cloud_cover": {"lt": 20}}  # meno del 20% di nuvole
)

items = search.get_all_items()
print(f"Trovate {len(items)} scene Sentinel-2")

# Stampiamo le prime 5 scene trovate
for i, item in enumerate(items[:5]):
    print(f"Scene {i+1}: {item.id}, Data: {item.datetime.strftime('%Y-%m-%d')}, Cloud cover: {item.properties['eo:cloud_cover']:.1f}%")

## 6. Selezione della scena migliore

In [ ]:
# Selezioniamo la scena con la minor copertura nuvolosa
best_item = min(items, key=lambda item: item.properties["eo:cloud_cover"])
print(f"Scena selezionata: {best_item.id}")
print(f"Data: {best_item.datetime.strftime('%Y-%m-%d')}")
print(f"Cloud cover: {best_item.properties['eo:cloud_cover']:.1f}%")

## 7. Download delle bande rilevanti di Sentinel-2

In [ ]:
# Definiamo le bande di interesse per il rilevamento delle aree umide
# Per Sentinel-2, usiamo le bande: Blue (B02), Green (B03), Red (B04), NIR (B08), SWIR1 (B11), SWIR2 (B12)
band_uris = {
    "B02": best_item.assets["B02"].href,  # Blue
    "B03": best_item.assets["B03"].href,  # Green
    "B04": best_item.assets["B04"].href,  # Red
    "B08": best_item.assets["B08"].href,  # NIR
    "B11": best_item.assets["B11"].href,  # SWIR1
    "B12": best_item.assets["B12"].href,  # SWIR2
}

# Leggiamo le bande richieste e le ritagliamo sulla nostra AOI
bands = {}
for band_name, uri in band_uris.items():
    print(f"Download banda {band_name}...")
    band_data = rioxarray.open_rasterio(uri)
    band_data = band_data.rio.reproject("EPSG:4326")
    band_data = band_data.rio.clip_box(
        minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max
    )
    bands[band_name] = band_data

print("Download delle bande completato.")

## 8. Visualizzazione dell'immagine RGB

In [ ]:
# Visualizziamo l'immagine RGB
rgb = np.stack([bands["B04"].values[0], bands["B03"].values[0], bands["B02"].values[0]], axis=-1)

# Normalizzazione per la visualizzazione
rgb_normalized = rgb.copy()
for i in range(3):
    p2, p98 = np.percentile(rgb[:,:,i], (2, 98))
    rgb_normalized[:,:,i] = np.clip((rgb[:,:,i] - p2) / (p98 - p2), 0, 1)

plt.figure(figsize=(12, 12))
plt.imshow(rgb_normalized)
plt.title(f"Immagine RGB Sentinel-2 {best_item.datetime.strftime('%Y-%m-%d')}")
plt.axis('off')
plt.show()

## 9. Definizione dell'architettura del modello CNN

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",      # Backbone
    encoder_weights="imagenet",   # Pesi pre-allenati
    in_channels=6,                # 6 bande Sentinel-2
    classes=1,                    # 1 classe (acqua)
    activation="sigmoid"          # Output binario
).to(device)

## 10. Preparazione dei dati di input

In [ ]:
# Preparazione dell'input per il modello
# Combiniamo tutte le bande in un unico array
input_data = np.stack([
    bands["B02"].values[0],  # Blue
    bands["B03"].values[0],  # Green
    bands["B04"].values[0],  # Red
    bands["B08"].values[0],  # NIR
    bands["B11"].values[0],  # SWIR1
    bands["B12"].values[0],  # SWIR2
], axis=0)

# Normalizziamo i dati
input_normalized = np.zeros_like(input_data, dtype=np.float32)
for i in range(input_data.shape[0]):
    p2, p98 = np.percentile(input_data[i], (2, 98))
    input_normalized[i] = np.clip((input_data[i] - p2) / (p98 - p2), 0, 1)

# Ridimensioniamo per processamento efficiente se necessario (opzionale)
# Per semplicità, manteniamo la risoluzione originale
original_shape = input_normalized.shape[1:]
print(f"Dimensioni dell'input: {input_normalized.shape}")

# Conversione a tensor e passaggio al device
input_tensor = torch.from_numpy(input_normalized).unsqueeze(0).to(device)  # [1, 6, H, W]
print("Dati preparati e caricati sul device.")

## 11. Applicazione del modello CNN

In [ ]:
# Applicazione del modello per il rilevamento delle aree umide
print("Applicazione del modello CNN per il rilevamento delle aree umide...")
with torch.no_grad():
    model.eval()
    water_prob = model(input_tensor)
    water_prob = water_prob.squeeze().cpu().numpy()

# Soglia per generare la maschera binaria (>0.5 = acqua)
water_mask = (water_prob > 0.5).astype(np.uint8)
print("Rilevamento completato.")

## 12. Visualizzazione e analisi dei risultati

In [ ]:
from matplotlib.colors import ListedColormap

# Visualizzazione dei risultati
plt.figure(figsize=(18, 12))

# Immagine RGB originale
plt.subplot(2, 2, 1)
plt.imshow(rgb_normalized)
plt.title("Immagine RGB Sentinel-2")
plt.axis('off')

# Indice NDWI (Normalized Difference Water Index)
# NDWI = (Green - NIR) / (Green + NIR)
ndwi = (bands["B03"].values[0] - bands["B08"].values[0]) / (bands["B03"].values[0] + bands["B08"].values[0] + 1e-6)
plt.subplot(2, 2, 2)
plt.imshow(ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
plt.colorbar(label='NDWI')
plt.title("NDWI (Normalized Difference Water Index)")
plt.axis('off')

# Mappa di probabilità dell'acqua
plt.subplot(2, 2, 3)
plt.imshow(water_prob, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(label='Probabilità')
plt.title("Mappa di probabilità delle aree umide")
plt.axis('off')

# Maschera binaria delle aree umide
water_cmap = ListedColormap(['none', 'blue'])
plt.subplot(2, 2, 4)
plt.imshow(rgb_normalized)
plt.imshow(water_mask, cmap=water_cmap, alpha=0.6)
plt.title("Rilevamento delle aree umide")
plt.axis('off')

plt.tight_layout()
plt.show()

## 13. Analisi quantitativa delle aree umide rilevate

In [ ]:
# Calcoliamo statistiche sulle aree umide rilevate
total_pixels = water_mask.size
water_pixels = np.sum(water_mask)
water_percentage = (water_pixels / total_pixels) * 100

print(f"Statistiche delle aree umide rilevate:")
print(f"- Numero totale di pixel: {total_pixels}")
print(f"- Pixel classificati come acqua: {water_pixels}")
print(f"- Percentuale di copertura d'acqua: {water_percentage:.2f}%")

# Calcolo dell'area approssimativa
# Nota: per un calcolo più preciso, bisognerebbe considerare la risoluzione spaziale e la proiezione
resolution_m = 10  # risoluzione Sentinel-2 per le bande RGB e NIR è 10m/pixel
water_area_m2 = water_pixels * resolution_m * resolution_m
water_area_km2 = water_area_m2 / 1_000_000

print(f"- Area stimata delle acque: {water_area_m2:,.0f} m² ({water_area_km2:.2f} km²)")

## 14. Salvataggio dei risultati

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# Salviamo la mappa di probabilità come GeoTIFF
output_probability = "water_probability_map.tif"
output_mask = "water_mask.tif"

# Otteniamo le coordinate e la trasformazione da una delle bande originali
blue_band = bands["B02"]
transform = from_bounds(
    blue_band.x.min().item(), 
    blue_band.y.min().item(), 
    blue_band.x.max().item(), 
    blue_band.y.max().item(), 
    water_prob.shape[1], 
    water_prob.shape[0]
)

# Salvataggio della mappa di probabilità
with rasterio.open(
    output_probability,
    'w',
    driver='GTiff',
    height=water_prob.shape[0],
    width=water_prob.shape[1],
    count=1,
    dtype=water_prob.dtype,
    crs=blue_band.rio.crs,
    transform=transform,
) as dst:
    dst.write(water_prob, 1)

# Salvataggio della maschera binaria
with rasterio.open(
    output_mask,
    'w',
    driver='GTiff',
    height=water_mask.shape[0],
    width=water_mask.shape[1],
    count=1,
    dtype=water_mask.dtype,
    crs=blue_band.rio.crs,
    transform=transform,
) as dst:
    dst.write(water_mask, 1)

print(f"Risultati salvati come:")
print(f"- {output_probability}")
print(f"- {output_mask}")